# Multimodal ViT — Demo
Zero-shot classification · Training curves · Embedding space

In [ ]:
import torch, json
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from torchvision import transforms

from models.text_encoder import Tokenizer
from models.model        import MultimodalModel

# ── Load checkpoint ───────────────────────────────────────────────
ckpt = torch.load('exp/best.pt', map_location='cpu')
cfg  = ckpt['cfg']

tok = Tokenizer(cfg['vocab_size'], cfg['max_seq_len'])
tok.load('exp/vocab.json')
cfg['vocab_size'] = len(tok)

model = MultimodalModel(cfg)
model.load_state_dict(ckpt['model'])
model.eval()

cat_names = json.load(open('data/categories.json'))
ann_val   = json.load(open('data/val/annotations.json'))

tfm = transforms.Compose([
    transforms.Resize((cfg['image_size'], cfg['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
def preprocess(img): return tfm(img.convert('RGB')).unsqueeze(0)

print(f'Loaded checkpoint from epoch {ckpt["epoch"]}.')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 1 — Training Curves

In [ ]:
def read_log(path):
    """Parse exp/train.log or exp/val.log into a dict of {key: [values]}."""
    data = {}
    for line in open(path):
        # format: 'YYYY-MM-DD HH:MM:SS  key=val  key=val ...'
        parts = line.split()[2:]   # skip timestamp
        for p in parts:
            k, v = p.split('=')
            data.setdefault(k, []).append(float(v))
    return data

train_data = read_log('exp/train.log')
val_data   = read_log('exp/val.log')

epochs = train_data['epoch']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, train_data['loss'], label='Train')
axes[0].plot(val_data['epoch'], val_data['loss'], label='Val')
axes[0].set_title('InfoNCE Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(val_data['epoch'], val_data['retrieval_acc'], color='green')
axes[1].set_title('Val Image→Text Retrieval Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Curves', fontweight='bold')
plt.tight_layout(); plt.show()

## 2 — Zero-Shot Classification

In [ ]:
sample = ann_val[0]
img    = Image.open(Path('data') / sample['file_path'])

# Build prompts from the GT classes present in this image + a few distractors
gt_class_ids = list({lbl for lbl in sample['labels'][:5]})
prompts  = [f'a photo of a {cat_names[i]}' for i in gt_class_ids]
prompts += ['a photo of a banana', 'a photo of a keyboard', 'a photo of a surfboard']

with torch.no_grad():
    probs = model.zero_shot(preprocess(img), tok.encode(prompts))

best = probs.argmax().item()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.imshow(img); ax1.axis('off')
ax1.set_title(f'Caption: {sample["caption"][:70]}', fontsize=8)
colors = ['crimson' if i == best else 'steelblue' for i in range(len(prompts))]
ax2.barh(prompts, probs.tolist(), color=colors)
ax2.set_xlim(0, 1); ax2.set_xlabel('Similarity probability')
ax2.set_title(f'Predicted: "{prompts[best]}"', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
# Run zero-shot on a small batch and show a grid
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
prompts_grid = [f'a photo of a {c}' for c in ['person','dog','car','bicycle','cat','horse','bird','truck']]

for ax, sample in zip(axes.flatten(), ann_val[:8]):
    img = Image.open(Path('data') / sample['file_path'])
    with torch.no_grad():
        probs = model.zero_shot(preprocess(img), tok.encode(prompts_grid))
    best  = probs.argmax().item()
    label = prompts_grid[best].replace('a photo of a ', '')
    conf  = probs[best].item()
    ax.imshow(img)
    ax.set_title(f'{label}  ({conf:.2f})', fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('Zero-Shot Classification Grid', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

## 3 — Cosine Similarity Matrix

In [ ]:
# Show how well image and text embeddings align for a small batch
n = 8
batch  = ann_val[:n]
images = torch.cat([preprocess(Image.open(Path('data') / s['file_path'])) for s in batch])
caps   = [s['caption'] for s in batch]

with torch.no_grad():
    img_emb = model.encode_image(images)
    txt_emb = model.encode_text(tok.encode(caps))
    sim     = (img_emb @ txt_emb.T).numpy()

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1)
ax.set_xticks(range(n)); ax.set_xticklabels([c[:30] for c in caps], rotation=45, ha='right', fontsize=7)
ax.set_yticks(range(n)); ax.set_yticklabels([c[:30] for c in caps], fontsize=7)
ax.set_title('Image–Text Cosine Similarity\n(diagonal = matching pairs)', fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

diag   = sim.diagonal().mean()
offdiag = (sim.sum() - sim.diagonal().sum()) / (n*n - n)
print(f'Mean diagonal (matching)     : {diag:.3f}')
print(f'Mean off-diagonal (non-match): {offdiag:.3f}')
print(f'Gap (higher = better)        : {diag - offdiag:.3f}')

## 4 — Run Inference from the command line

In [ ]:
# Run from a terminal (or uncomment the line below to run directly in the notebook)
#
# python inference.py \
#     --image data/Images/000000001000.jpg \
#     --prompts "a photo of a dog" "a photo of a cat" "a photo of a car"

import subprocess
from pathlib import Path

# Pick the first val image as a demo
sample     = ann_val[0]
image_path = Path('data') / sample['file_path']
prompts    = ['a photo of a person', 'a photo of a dog', 'a photo of a car',
              'a photo of a bicycle', 'a photo of a cat']

cmd = ['python', 'inference.py', '--image', str(image_path), '--prompts'] + prompts
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)